# Solar Panel Detection

This section will show how to detect solar panels in a raster dataset.

In [ ]:
# import geoai package

import geoai

# Download the sample data

raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/solar_panels_davis_ca.tif"
raster_path = geoai.download_file(raster_url)

geoai.print_raster_info(raster_path)


In [ ]:
# Move the downloaded files to the target folder

import os
import shutil
from pathlib import Path

# ==== INPUTS ====

target_folder = "data/geoai/solar-panel-detection"  # Enter your target folder path here
datasets = ["solar_panels_davis_ca.tif"]  # Add dataset names as a list
source_dir = Path(".")  # Current location of datasets

# ==== SCRIPT ====

target_path = Path(target_folder)
target_path.mkdir(parents=True, exist_ok=True)
for dataset_name in datasets:
    source_path = source_dir / dataset_name
    
    if not source_path.exists():
        print(f"Dataset '{dataset_name}' not found in {source_dir}")
        continue
    
    if source_path.is_file():
        files_to_move = [source_path]
    else:
        files_to_move = list(source_path.parent.glob(f"{dataset_name}.*"))
    
    for file_path in files_to_move:
        dest = target_path / file_path.name
        shutil.move(str(file_path), str(dest))
        print(f"Moved: {file_path.name} -> {target_folder}")
        
print("Done!")

In [ ]:
# Visualize data
raster = "data/geoai/solar-panel-detection/solar_panels_davis_ca.tif"

geoai.view_raster(raster)

In [ ]:
# Initialize the model

detector = geoai.SolarPanelDetector()

In [ ]:
# Generate masks

output_path = "data/geoai/solar-panel-detection/solar_panel_masks.tif"

masks_path = detector.generate_masks(
    raster,
    output_path=output_path,
    confidence_threshold=0.4,
    mask_threshold=0.5,
    min_object_area=100,
    overlap=0.25,
    chip_size=(400, 400),
    batch_size=4,
    verbose=False,
)

In [ ]:
# Visualize mask

geoai.view_raster(
    output_path,
    indexes=[2],
    colormap="autumn",
    layer_name="Solar Panels",
    basemap=raster_url,
)

In [ ]:
# Vectorize mask

gdf = geoai.orthogonalize(
    input_path=masks_path, output_path="data/geoai/solar-panel-detection/solar_panel_masks.geojson", epsilon=0.2
)

In [ ]:
# Visualize initial result 

geoai.view_vector_interactive(gdf, tiles=raster_url)

In [ ]:
# Calculate geometric properties

gdf = geoai.add_geometric_properties(gdf)
gdf.head()
print(len(gdf))
geoai.view_vector_interactive(gdf, column="elongation", tiles=raster_url)

In [ ]:
# Filter results

gdf_filter = gdf[(gdf["elongation"] < 10) & (gdf["area_m2"] > 5)]
print(len(gdf_filter))

In [ ]:
# Visualize final results

geoai.view_vector_interactive(gdf_filter, column="area_m2", tiles=raster)

In [ ]:
geoai.view_vector_interactive(
    gdf_filter, style_kwds={"color": "red", "fillOpacity": 0}, tiles=raster
)

In [ ]:
gdf_filter["area_m2"].hist()

In [ ]:
gdf_filter["area_m2"].describe()

In [ ]:
gdf_filter["area_m2"].sum()

In [ ]:
# Save the final result

gdf_filter.to_file("data/geoai/solar-panel-detection/solar_panels_dubai.geojson")